# DeepPEF Training — Final Pipeline

This notebook runs the full training pipeline for the thesis.

**Prerequisites:** Upload your data to Google Drive at `My Drive/DeepPEF_data/`:
- `training_data/` (368 protein folders with .pt files)
- `mutation_files/` (CSV files)
- `ThermoMPNN/mega_test.csv`
- `Pnas_filtering/train_proteins.csv` and `pnas_mutations.csv`

**To upload data from GPU machine:**
```bash
# On GPU machine, tar the data:
cd /home/nissimb/workspace/DeepPEF
tar czf deepef_data.tar.gz data/MsDs/ data/ThermoMPNN/ data/Processed_K50_dG_datasets/Pnas_filtering/
# Then upload deepef_data.tar.gz to Google Drive
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and install deps
!git clone https://github.com/shaharec/DeepPEF.git /content/DeepPEF
%cd /content/DeepPEF
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install torch-geometric scipy scikit-learn pandas tqdm wandb biopython fair-esm -q

In [ ]:
# Extract data from Drive (if uploaded as tar)
import os

DATA_TAR = '/content/drive/MyDrive/DeepPEF_data/deepef_data.tar.gz'
if os.path.exists(DATA_TAR):
    !tar xzf {DATA_TAR} -C /content/DeepPEF/
    print('Data extracted from tar')
elif os.path.exists('/content/drive/MyDrive/DeepPEF_data/training_data'):
    # Symlink if data is already in Drive as folders
    !mkdir -p data/MsDs
    !ln -sf /content/drive/MyDrive/DeepPEF_data/training_data data/MsDs/training_data
    !ln -sf /content/drive/MyDrive/DeepPEF_data/mutation_files data/MsDs/mutation_files
    !mkdir -p data/ThermoMPNN
    !ln -sf /content/drive/MyDrive/DeepPEF_data/ThermoMPNN/mega_test.csv data/ThermoMPNN/mega_test.csv
    !ln -sf /content/drive/MyDrive/DeepPEF_data/ThermoMPNN/mega_train.csv data/ThermoMPNN/mega_train.csv
    !mkdir -p data/Processed_K50_dG_datasets/Pnas_filtering
    !ln -sf /content/drive/MyDrive/DeepPEF_data/Pnas_filtering/train_proteins.csv data/Processed_K50_dG_datasets/Pnas_filtering/train_proteins.csv
    !ln -sf /content/drive/MyDrive/DeepPEF_data/Pnas_filtering/pnas_mutations.csv data/Processed_K50_dG_datasets/Pnas_filtering/pnas_mutations.csv
    print('Data symlinked from Drive')
else:
    print('ERROR: Upload data to Google Drive first!')
    print('Expected: /content/drive/MyDrive/DeepPEF_data/deepef_data.tar.gz')

In [ ]:
# Verify data
import torch, os
td = './data/MsDs/training_data'
proteins = [d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d))]
print(f'Total proteins: {len(proteins)}')

# Check esmif_enc.pt
has_esmif = [p for p in proteins if os.path.exists(os.path.join(td, p, 'esmif_enc.pt'))]
print(f'Have ESM-IF1 features: {len(has_esmif)}/{len(proteins)}')

# Check emb.pt
has_emb = [p for p in proteins if os.path.exists(os.path.join(td, p, 'emb.pt'))]
print(f'Have ProtT5 embeddings: {len(has_emb)}/{len(proteins)}')

# Check test split
import pandas as pd
tm = pd.read_csv('./data/ThermoMPNN/mega_test.csv')
test_names = tm['WT_name'].str.replace('.pdb', '', regex=False).unique().tolist()
test_found = [p for p in test_names if p in proteins]
print(f'Test proteins: {len(test_found)}/{len(test_names)}')
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Generate ESM-IF1 encoder features if missing
if len(has_esmif) < len(proteins):
    print(f'Generating ESM-IF1 features for {len(proteins) - len(has_esmif)} proteins...')
    !python data_creation/generate_esmif_encoder_features.py
else:
    print('ESM-IF1 features already generated for all proteins')

## Step 1: ProtT5 Baseline (5-seed ensemble)
This is our known-good configuration: PCC ~0.52 per seed, ensemble ~0.55-0.57

In [ ]:
%%time
# ProtT5 baseline — 5 seeds
import subprocess

os.environ['WANDB_MODE'] = 'disabled'
seeds = [42, 123, 456, 789, 1337]

for seed in seeds:
    print(f'\n{"="*50}')
    print(f'ProtT5 baseline — seed {seed}')
    print(f'{"="*50}')
    !python Megascale-fineTuning/pnas_train.py \
        --model_name baseline_prott5_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --emb_type prott5 \
        --mini_batch_size 64 \
        --emb_projection none

## Step 2: dual_esmif WITH Projection (5-seed ensemble)
ProtT5 (1024) + ESM-IF1 (512) = 1536 dim → projected to 16 before GNN.
This keeps batch_size=64 (no OOM) while adding structural features.

In [ ]:
%%time
# dual_esmif with MLP projection — 5 seeds
for seed in seeds:
    print(f'\n{"="*50}')
    print(f'dual_esmif + projection — seed {seed}')
    print(f'{"="*50}')
    !python Megascale-fineTuning/pnas_train.py \
        --model_name v4_dual_esmif_proj_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --emb_type dual_esmif \
        --mini_batch_size 64 \
        --emb_projection mlp

## Step 3: ESM-IF1 Only with Projection (comparison)
Pure structural features — how much does structure alone contribute?

In [ ]:
%%time
# ESM-IF1 only + projection — 5 seeds
for seed in seeds:
    print(f'\n{"="*50}')
    print(f'ESM-IF1 only + projection — seed {seed}')
    print(f'{"="*50}')
    !python Megascale-fineTuning/pnas_train.py \
        --model_name v4_esmif_only_proj_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --emb_type esmif_enc \
        --mini_batch_size 64 \
        --emb_projection mlp

## Results Summary

In [ ]:
# Collect and display results
import glob
import json

model_dir = './Megascale-fineTuning/models'
results = {}

for folder in sorted(os.listdir(model_dir)):
    best_path = os.path.join(model_dir, folder, 'best_model.pt')
    if os.path.exists(best_path):
        results[folder] = os.path.getmtime(best_path)

print('Models trained:')
for name in sorted(results.keys()):
    print(f'  {name}')

print(f'\nTotal: {len(results)} models')
print('\nCheck PCC results in the training output above.')
print('Look for lines like: "Training completed with Best Pearson Correlation: X.XXXX"')

In [ ]:
# Save models back to Google Drive
!tar czf /content/drive/MyDrive/DeepPEF_data/trained_models.tar.gz Megascale-fineTuning/models/
print('Models saved to Google Drive')